In [1]:
import hydra
from omegaconf import DictConfig
from hydra import initialize, compose

import pandas as pd
import os

## Training the model

In [2]:
with initialize(config_path='../configs'):
    cfg = compose(config_name="config")

/tmp/ipykernel_2211465/2532472650.py:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path='../configs'):


In [ ]:
## The following is the configuration file that you need to use for the experiment. Here is only for information. The command to run the experiment is in the next cell.

from datetime import datetime

# Get current date and time
now = datetime.now()

# Format it as a string
timestamp_str = now.strftime("%Y-%m-%d_%H:%M:%S")

cfg.general.name = 'luna' + '_' + timestamp_str
train_test_data_folder = 'train_test_split_1'

cfg.distribute.gpus_per_node=[1]
# cfg.general.wandb='disabled'
cfg.general.debug = True

cfg.dataset.maximum_graph_size.train=1000
cfg.dataset.maximum_graph_size.test=1000
cfg.train.batch_size=4
cfg.model.hidden_dims.cell_image_dimensions=256
cfg.model.hidden_dims.num_heads=16

# Use the setting to quickly check the model
cfg.validation.check_val_every_n_epochs=1
cfg.validation.save_model_every_n_epochs=1
cfg.train.n_epochs=4
cfg.model.diffusion_steps=2

cfg.dataset.gene_columns_start = 13
cfg.dataset.gene_columns_end = 360


run_directory = '/home/anagupta/luna/runs' + '/' + 'run_'+ timestamp_str
if not os.path.exists(run_directory):
    os.makedirs(run_directory)
cfg.general.local_saved_path = run_directory + '/train_results'
cfg.test.save_dir = run_directory + '/test_results' # Change this to the directory where you want to save the results

data_directory = '/home/anagupta/luna/' + train_test_data_folder
cfg.dataset.train_data_path = data_directory + '/train_data.csv' # Change this to the path of the train csv file
cfg.dataset.test_data_path = data_directory + '/test_data.csv' # Change this to the path of the test csv file
cfg.dataset.slice_images_path = data_directory + '/slice_images' # Change this to the path of the slice images
cfg.dataset.train_cell_images_path = data_directory + '/train_cell_images' # Change this to the path of the train cell images
cfg.dataset.test_cell_images_path = data_directory + '/test_cell_images' # Change this to the path of the test cell images

cfg.dataset.dataset_name = 'luna_with_cell_images' if cfg.dataset.train_cell_images_path else 'luna_without_cell_images'

from omegaconf import OmegaConf

# Save the cfg configuration file
OmegaConf.save(cfg, run_directory + '/config.yaml')

In [ ]:
output_path =  dir + '/output.txt'
!python3 /home/anagupta/luna/LUNA/main.py --config-path=$root_directory --config-name=config.yaml > $output_path

## Testing the model

In [2]:
dir = '/mlbio_scratch/anagupta/luna/runs/baseline/2025-06-11_12-44-38/.hydra'

relative_dir=os.path.relpath(dir, os.getcwd())

with initialize(config_path=relative_dir):
    cfg = compose(config_name="config")

name = cfg.general.name + '_test'
output_path =  dir + '/output_test2.log'

os.environ['HYDRA_FULL_ERROR'] = '1'
os.environ['name'] = name
os.environ['dir'] = dir
os.environ['output_path'] = output_path

print("Check Output logs: ", output_path)

!python3 /home/anagupta/luna/LUNA/main.py --config-path=$dir --config-name=config.yaml general.mode='test_only' general.name=$name > $output_path 2>&1

/tmp/ipykernel_2215631/1789050412.py:5: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path=relative_dir):


Check Output logs:  /mlbio_scratch/anagupta/luna/runs/baseline/2025-06-11_12-44-38/.hydra/output_test2.log
